# K-Means Clustering — Antarctica (Ross Sea)

Region: 160–220°E (0-360 convention), 80–60°S — covers both R1 (160–180°E) and R2 (180–140°W)  
without splitting at the antimeridian.

Features: IWP, LWP, AOT — ocean pixels only (`land_flag < 0.1`), log1p-normalised.

In [ ]:
from pystac_client import Client
import fsspec
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import requests
from IPython.display import Image, display
import os
import pathlib

from scipy import stats
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

import numpy as np

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

In [ ]:
def cluster_variance(n, x):
    """Compute K-means inertia for k = 1 … n (elbow method).

    Parameters
    ----------
    n : int
        Maximum number of clusters to evaluate.
    x : array-like of shape (n_samples, n_features)
        Normalised feature matrix.

    Returns
    -------
    variances : list of float
        Inertia (sum of squared distances to nearest centroid) for each k.
    K : list of int
        Corresponding k values [1, 2, …, n].
    """
    variances = []
    K = list(range(1, n + 1))
    for k in K:
        model = KMeans(n_clusters=k, random_state=82, verbose=0).fit(x)
        variances.append(model.inertia_)
    return variances, K

In [ ]:
# Antarctica (Ross Sea) — lon converted to 0-360 to avoid splitting at the antimeridian.
# R1 (160-180°E) and R2 (180-140°W = 180-220° in 0-360) become one contiguous box.
# land_flag < 0.1 keeps only observations where ≥90% of the 1-min window is over ocean.
ds = xr.open_dataset("challenge_1min_numerical_AN.nc")

lon360 = ds.longitude % 360
ds = ds.where(
    (ds.latitude > -80) & (ds.latitude < -60) &
    (lon360 > 160) & (lon360 < 220) &
    (ds.land_flag < 0.1),
    drop=True
)
ds

In [ ]:
# Cluster on IWP, LWP, AOT only — land_flag was a surface property, not a cloud-aerosol feature
x = ds.drop_vars(['latitude', 'longitude', 'land_flag']).to_array().transpose("time", "variable").values

# log1p compresses the heavy tail of cloud/aerosol retrievals (they are roughly log-normal)
# so that a single large storm does not dominate the feature space
x = np.log1p(x)
x = (x - np.min(x, axis=0)) / (np.max(x, axis=0) - np.min(x, axis=0))
x

In [ ]:
variances, K = cluster_variance(20, x)
plt.plot(K, variances)
plt.ylabel("Inertia ( Total Distance )")
plt.xlabel("K Value")
plt.xticks(K)
plt.title("Elbow plot — Antarctica")
plt.show()

In [ ]:
ds

In [ ]:
k = 5

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(x)

In [ ]:
# Column indices in x after dropping lat, lon, land_flag.
# Order follows dataset variable order: ice_water_path, liquid_water_path, aerosol_optical_thickness_355nm
feature_cols = {
    'iwp': 0,
    'lwp': 1,
    'aot': 2,
}

In [ ]:
def plot_kmeans(x, varX, varY, y_kmeans, kmeans_model, ax=None):
    """Scatter plot of two normalised features coloured by cluster label.

    Parameters
    ----------
    x : ndarray of shape (n_samples, n_features)
        Normalised feature matrix.
    varX, varY : str
        Feature names (keys of ``feature_cols``) for the x and y axes.
    y_kmeans : ndarray of shape (n_samples,)
        Cluster labels returned by KMeans.fit_predict.
    kmeans_model : KMeans
        Fitted KMeans instance (used to draw centroid markers).
    ax : matplotlib.axes.Axes, optional
        Axes to draw on; a new figure is created when None.
    """
    if ax is None:
        fig, ax = plt.subplots()

    ax.scatter(x[:, feature_cols[varX]], x[:, feature_cols[varY]],
               c=y_kmeans, s=10, cmap='jet', alpha=0.5)

    centers = kmeans_model.cluster_centers_
    ax.scatter(centers[:, feature_cols[varX]], centers[:, feature_cols[varY]],
               c='red', s=50, alpha=0.75, marker='X', label='Centroids')

    n_clusters = len(centers)
    for i in range(n_clusters):
        ax.text(centers[i, feature_cols[varX]] * 1.1,
                centers[i, feature_cols[varY]] * 1.1,
                s=str(i + 1), c='red')

    ax.set_title("K-Means clustering — Antarctica")
    ax.set_xlabel(varX)
    ax.set_ylabel(varY)
    ax.legend()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_kmeans(x, 'aot', 'iwp', y_kmeans, kmeans, ax=axes[0])
plot_kmeans(x, 'aot', 'lwp', y_kmeans, kmeans, ax=axes[1])
plot_kmeans(x, 'iwp', 'lwp', y_kmeans, kmeans, ax=axes[2])

In [ ]:
kmeans.cluster_centers_

In [ ]:
plt.scatter(ds['longitude'].values, ds['latitude'].values, c=y_kmeans, cmap='jet')
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.title('Cluster spatial distribution — Antarctica')
plt.colorbar(label='Cluster')

## Cross-reference with AC\_\_TC\_\_2B synergetic classification

Each cluster is characterised by the distribution of the synergetic target classification (`stc_*`) at eight altitude levels (2.5–20 km). Histograms are colour-coded by class type.

In [ ]:
# Synergetic class range and colors
plot_range = range(-1, 35)
plot_colors = [
    "#c5c9c7", "#a2653e", "#ffffff", "#ff474c", "#0504aa", "#009337",
    "#840000", "#042e60", "#d8dcd6", "#ffff84", "#f5bf03", "#f97306",
    "#ff000d", "#5539cc", "#2976bb", "#0d75f8", "#014182", "#017b92",
    "#06b48b", "#aaff32", "#6dedfd", "#01f9c6", "#7bc8f6", "#d7fffe",
    "#a2cffe", "#04d9ff", "#7a9703", "#b2996e", "#ffbacd", "#d99b82",
    "#947e94", "#856798", "#ac86a8", "#59656d", "#76424e", "#363737"
]
labels = [
    "-1: unknown"," 0: ground"," 1: clear"," 2: possible rain (clutter)"," 3: possible snow (clutter)",
    " 4: possible cloud (clutter)"," 5: heavy rain"," 6: heavy mixed-phase precip",
    " 7: no rain/ice (possible liquid)"," 8: liquid cloud"," 9: drizzling liquid cloud",
    "10: warm rain","11: cold rain","12: melting snow","13: snow (possible liquid)",
    "14: snow (no liquid)","15: rimed snow (possible liquid)",
    "16: rimed snow + supercooled liquid","17: snow + supercooled liquid",
    "18: supercooled liquid","19: ice cloud (possible liquid)",
    "20: ice + supercooled liquid","21: ice cloud (no liquid)",
    "22: stratospheric ice","23: STS (PSC Type I)","24: NAT (PSC Type II)","25: insects",
    "26: dust","27: sea salt","28: continental pollution","29: smoke","30: dusty smoke",
    "31: dusty mix","32: stratospheric ash","33: stratospheric sulfate","34: stratospheric smoke"
]
cmap = mcolors.ListedColormap(plot_colors)
bounds = list(plot_range) + [34]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

In [ ]:
# Load synergetic labels with the same bbox + ocean filter applied to ds above
ds_tc = xr.open_dataset("challenge_1min_complete_AN.nc")

lon360_tc = ds_tc.longitude % 360
ds_tc = ds_tc.where(
    (ds_tc.latitude > -80) & (ds_tc.latitude < -60) &
    (lon360_tc > 160) & (lon360_tc < 220) &
    (ds_tc.land_flag < 0.1),
    drop=True
)
ds_tc

In [ ]:
row_labels = [
    "stc_2500", "stc_5000", "stc_7500", "stc_10000",
    "stc_12500", "stc_15000", "stc_17500", "stc_20000"
]
col_labels = [f"Cluster {i+1}" for i in range(k)]

fig, axes = plt.subplots(len(row_labels), k, figsize=(24, 20))

for i, label in enumerate(row_labels):
    row_axes = axes[i, :]
    y_pos = np.mean([ax.get_position().y0 + ax.get_position().height / 2 for ax in row_axes])
    x_pos = row_axes[0].get_position().x0 - 0.05
    fig.text(x_pos, y_pos, label, va='center', ha='right', fontsize=14)

    for cluster_idx in range(k):
        n, bins, patches = axes[i, cluster_idx].hist(
            ds_tc[label][y_kmeans == cluster_idx],
            bins=bounds)
        for patch, color in zip(patches, plot_colors):
            patch.set_facecolor(color)

for j, label in enumerate(col_labels):
    col_axes = axes[:, j]
    x0 = min(ax.get_position().x0 for ax in col_axes)
    x1 = max(ax.get_position().x0 + ax.get_position().width for ax in col_axes)
    x_center = (x0 + x1) / 2
    top_ax = axes[0, j]
    y_top = top_ax.get_position().y0 + top_ax.get_position().height
    fig.text(x_center, y_top + 0.02, label, ha='center', va='bottom', fontsize=16)